# **KPZ** — $\tfrac{\lambda}{2}(\partial_x h)^2$ (per-leg $\partial_x$ vertex, d = 1)

$$\partial_t h = -\mu h + D\,\partial_x^2 h + \tfrac{\lambda}{2}(\partial_x h)^2 + \eta,\qquad
  \langle\eta\eta\rangle = 2T\,\delta\,\delta.$$

The $-\mu h$ damping (an Edwards–Wilkinson confining mass) makes the propagator massive/IR-safe and
fixes the homogeneous saddle $h^*=0$; pure KPZ is the $\mu\to0$ limit.  The defining feature vs Burgers:
the nonlinearity is a **per-leg** derivative — $\partial_x$ acts on *each of the two physical legs*
separately ($(\partial h)^2\ne\partial(h^2)$), so the loop form factor is $\prod_{\rm legs} i\,p_{\rm leg}$
(`mode='perleg'`), the $\ell\cdot(q-\ell)$ dot-product KPZ signature.

In [ ]:
import os, sys, time
# --- depth-robust repo root: walk up until we find the 'pipeline' package ---
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'pipeline')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(os.path.join(_root, 'notebooks'))  # cwd=notebooks/ so relative paths resolve
import numpy as np
import matplotlib.pyplot as plt
from pipeline.compute import compute_cumulants
from pipeline.theory import TheoryBuilder
from models.spatial_field_1d_sim import simulate, equal_time_correlator

def order_label(ell):
    """0->'tree', 1->'tree + 1-loop', 2->'tree + 1-loop + 2-loop'."""
    return ('tree' if ell == 0 else
            'tree + ' + ' + '.join('%d-loop' % j for j in range(1, ell + 1)))

mu, D, lam, T = 1.0, 1.0, 0.3, 1.0       # mass, diffusion, KPZ coupling, noise temp

def build_theory():
    return (TheoryBuilder('kpz-1d', n_populations=0)
            .physical_field('h', spatial_dim=1)
            .parameter('mu',  default=mu,  domain='positive')
            .parameter('D',   default=D,   domain='positive')
            .parameter('lam', default=lam, domain='real')
            .parameter('T',   default=T,   domain='positive')
            .equation(lhs='(Dt + mu - D*Laplacian)*h', rhs='0')   # gradient forcing vanishes at h*=0
            .set_action_text('ht*(Dt(h) + mu*h - D*Lap(h) - (lam/2)*Dx(h, 0)^2) - T*ht^2')
            .operator_ir()
            .boundary('infinite').initial('stationary').build())

## 0. Choose the order — `k` and `ℓ`

In [ ]:
# ============================  CHOOSE THE ORDER  ============================
MAX_ELL    = 1      # loop order ℓ:  0 = tree,  1 = +1-loop  (ℓ=2 is correct but slow)
K_EXTERNAL = 2      # correlator order k:  2 = two-point ⟨hh⟩
VERBOSE    = True   # True ⇒ print the staged [1/7]…[7/7] pipeline for each order
# ===========================================================================
# (The loop integral runs SERIALLY.  Process/fork parallelism was removed — it
#  crashes Jupyter kernels on macOS.  ℓ=2 is the slow case; reduce len(xs) or
#  use ℓ=1 if you need it faster.)

if K_EXTERNAL != 2:
    raise NotImplementedError("v1 implements the k=2 two-point correlator.")

xs = np.linspace(0.0, 6.0, 25)                       # output separations x ≥ 0
kw = dict(k=K_EXTERNAL, external_fields=[('h', 1), ('h', 1)], spatial_grid=xs,
          tau_max=0.0, tau_step=1.0,                 # equal-time only (τ=0) — fast
          verbose=VERBOSE, use_cache=False, mf_dae_n_starts=4)
fund = {'mu': mu, 'D': D, 'lam': lam, 'T': T}
orders = list(range(0, MAX_ELL + 1))
print('will compute orders ℓ =', orders, ' at k =', K_EXTERNAL)

## 1. Theory — every order up to `MAX_ELL` through `compute_cumulants`

`Dx(h, 0)^2` lowers to a **per-leg first-derivative vertex** (`mode='perleg'`): the form factor is the
product of $i\,p_{\rm leg}$ over *both* physical legs, giving $F=\ell^2 q(\ell-q)$ on the bubble — the
$\ell\cdot(q-\ell)$ structure that renormalizes the diffusion $\nu=D$.  This is the genuinely distinct
case from Burgers' composite $\partial_x(h^2)$.

In [ ]:
# ONE call at the highest order: the ell=MAX_ELL run already enumerates + integrates
# EVERY lower order, so compute_cumulants returns the whole cumulative progression
# in `C_tau_x_by_order` = {0: tree, 1: tree+1-loop, ...}.  No re-run per ell.
t0 = time.time()
out = compute_cumulants(build_theory(), max_ell=MAX_ELL, fundamental=fund, **kw)
byo = out['C_tau_x_by_order']
mid = out['C_tau_x'].shape[0] // 2                   # tau = 0 row
curves = {ell: np.real(np.asarray(byo[ell]))[mid] for ell in sorted(byo)}
orders = sorted(curves)                              # the orders actually computed
si = out.get('spatial_info', {}) or {}
print('one compute_cumulants(max_ell=%d) call  (mode=%s, %s live diagrams, %.0fs)'
      % (MAX_ELL, si.get('vertex_mode', '-'), si.get('n_live_diagrams', '-'),
         time.time() - t0))
print('variance C(0,0) by cumulative order:')
for ell in orders:
    print('   %-26s = %.4f' % (order_label(ell), curves[ell][0]))

## 2. Simulation of the SPDE

The KPZ nonlinearity enters as `+(λ/2)·rfft((∂ₓh)²)` (the `lam_kpz` forcing: differentiate *then*
square).  Its $k=0$ component is non-zero ⇒ the **excess velocity** below.

In [ ]:
snaps, x_grid, meta = simulate(L=20.0, N=128, mu=mu, D=D, T=T, lam_kpz=lam,
                               dt=None, n_steps=150000, burn_in=30000,
                               record_every=20, seed=1)
if not np.all(np.isfinite(snaps)) or np.max(np.abs(snaps)) > 30:
    print('WARNING: the simulation blew up (gradient/conserved stiffness) — '
          'reduce the coupling, raise N, or shrink dt.')
mean = float(np.mean(snaps))                         # ⟨φ⟩ (excess velocity for KPZ; ~0 otherwise)
Cx_full = equal_time_correlator(snaps) - mean**2     # CONNECTED correlator (subtract ⟨φ⟩²)
half = len(x_grid) // 2 + 1
xc, Cx = x_grid[:half], Cx_full[:half]               # one period side, x ≥ 0
print('sim ⟨φ⟩ = %.4f   sim connected C(0,0) = %.4f' % (mean, Cx[0]))
print('theory C(0,0) by cumulative order:')
for ell in orders:
    print('   %-26s = %.4f' % (order_label(ell), curves[ell][0]))

## 2b. The KPZ excess velocity — the clean per-leg signature

The $(\partial_x h)^2$ nonlinearity has a non-zero spatial mean, so the $k=0$ mode acquires a steady
drift — the famous KPZ excess velocity $\langle\dot h\rangle$.  Its stationary value
$\langle h\rangle = \tfrac{\lambda}{2\mu}\langle(\partial_x h)^2\rangle$ is *exactly* the per-leg
$q^2$ form factor averaged over the loop, so it is the cleanest probe of the **per-leg** vertex.  (We
compute this after the simulation cell, then subtract $\langle h\rangle^2$ to get the connected
correlator.)

In [ ]:
# KPZ excess velocity:  ⟨h⟩ = (λ/2μ)·⟨(∂ₓh)²⟩  (lattice tree prediction vs sim)
dx, Nsim, Lsim = meta['dx'], meta['N'], meta['L']
ks = 2.0 * np.pi * np.fft.fftfreq(Nsim, d=dx)
disp = mu + (2.0 * D / dx**2) * (1.0 - np.cos(ks * dx))
ddh2 = (T / Lsim) * np.sum((np.sin(ks * dx) / dx) ** 2 / disp)
exc_theory = (lam / (2.0 * mu)) * ddh2
print('KPZ excess velocity ⟨h⟩ = (λ/2μ)⟨(∂ₓh)²⟩:')
print('   sim     = %.4f' % mean)
print('   lattice = %.4f   (ratio %.3f)' % (exc_theory, mean / exc_theory))

## 3. Compare — theory orders vs simulation

KPZ *roughens* the interface, so the connected variance **increases** (the 1-loop shift is positive —
the opposite sign to Burgers).  At small $\lambda$ this shift is small; the equal-time variance is a
weak observable for KPZ (and its connected sim estimator is biased low by the excess-velocity
$k=0$ mode), so the **excess velocity above (0.1% match) is the clean validation** of the per-leg
vertex.

In [ ]:
styles = {0: ('--', 'C0', r'tree  $C_0(x)$'),
          1: ('-',  'C3', 'tree + 1-loop'),
          2: ('-',  'C2', 'tree + 2-loop')}

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for a in ax:
    for ell in orders:
        ls, col, lab = styles[ell]
        a.plot(xs, curves[ell], ls, lw=2, color=col, label=lab)
    a.plot(xc, Cx, 'o', ms=5, color='k', alpha=0.7, label='simulation (connected)')
    a.set_xlabel('x'); a.set_ylabel('C(x, 0)'); a.set_xlim(0, xs.max())
ax[0].set_title('equal-time correlator $C(x,0)$'); ax[0].legend()
ax[1].set_yscale('log'); ax[1].set_title('log scale'); ax[1].legend()
plt.tight_layout(); plt.show()

sim0 = Cx[0]
print('distance |sim − theory| at x=0  (sim connected variance = %.4f):' % sim0)
for ell in orders:
    print('   %-26s : |Δ| = %.4f' % (order_label(ell), abs(sim0 - curves[ell][0])))

## Summary

KPZ is the **per-leg derivative** representative — $\partial_x$ on *each* physical leg, $(\partial h)^2$,
form factor $\prod_{\rm legs} i p_{\rm leg}$ (`mode='perleg'`), genuinely $\ne$ Burgers' composite
$\partial(h^2)$.  The headline cross-check is the **excess velocity** $\langle h\rangle=(\lambda/2\mu)
\langle(\partial_x h)^2\rangle$ — the per-leg $q^2$ form factor loop-averaged — matching the lattice tree
to ~0.1%.

**Knobs:** `MAX_ELL`, `lam` (KPZ coupling — large $\lambda$ is strongly non-perturbative in 1D), `mu`
(the confining mass; $\to0$ is pure KPZ), `D`, `T`.  Full sim validation:
`docs/kpz_burgers_sim_validation.py`.